# Lesson 04 Lab — Why Data Movement Can Cost More Than Arithmetic

**Puzzle:** If a GPU can execute enormous arithmetic throughput, why can a simple elementwise operation remain slow?

This notebook retains one complete RTX 5090 execution.


## Why this matters

An operation cannot use a compute unit until its operands arrive. Moving data activates wires, buffers, routing, tags, controllers, and storage arrays across distance; a multiply-add reusing values already near an execution unit may perform far more useful arithmetic per transferred byte. Arithmetic intensity—operations divided by bytes moved—connects an algorithm to this physical distinction.


## 0. Predict before running

1. Predict which workload is closer to a bandwidth ceiling.
2. Compute vector-add intensity assuming two reads and one write.
3. Name one reason achieved values stay below either theoretical roof.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The Roofline bound is `min(peak_compute, bandwidth × arithmetic_intensity)`. Vector addition has low intensity because it reads two arrays and writes one for one addition. A large matrix multiplication reuses each tile and can have much higher intensity. The notebook measures a vector operation and BF16 GEMM through the same CUDA-event helper, reports effective bandwidth or TFLOP/s, and keeps analytical bytes/FLOPs beside time. The two metrics are not compared as if they were interchangeable.

- Performance needs both an operation count and a byte count.
- Low intensity puts the bandwidth roof below the compute roof.
- Tiling and fusion help when they remove or amortize traffic, not merely because they add code.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["algorithm"] --> B["FLOPs"]
  A --> C["bytes moved"]
  B --> D["arithmetic intensity"]
  C --> D
  D --> E["bandwidth or compute roof"]
```


## 3. Inspect the visual boundary

This lesson is driven by a Mermaid mechanism map and executable measurements.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 4
LESSON_TITLE = 'Why Data Movement Can Cost More Than Arithmetic'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260817
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | elementwise `a + b` over a large tensor |
| Candidate | square BF16 matrix multiplication |
| Held constant | GPU, warm-up, repetitions, dtype, and event timing |
| Measurements | arithmetic intensity, median latency, effective GB/s, and TFLOP/s |
| Evidence | `pytorch-gpu` |

**Experiment:** Measure a low-intensity vector expression and a reuse-heavy BF16 GEMM on one GPU.


## 6. Inspect the code

The vector path counts compulsory tensor traffic; the GEMM path uses `2MNK` FLOPs and input/output bytes as an algorithmic intensity estimate. Library internals and cache traffic are left as measured follow-ups.

Do not run until the code matches the frozen table.


In [2]:
dtype = torch.bfloat16
n = 2**26
a = torch.randn(n, device=DEVICE, dtype=dtype)
b = torch.randn(n, device=DEVICE, dtype=dtype)
c = torch.empty_like(a)
vector_samples = cuda_samples(lambda: torch.add(a, b, out=c), repeats=25)
vector_median = statistics.median(vector_samples)
vector_bytes = 3 * n * a.element_size()
vector_gbps = vector_bytes / (vector_median / 1e3) / 1e9
vector_intensity = n / vector_bytes

m = 2048
x = torch.randn((m, m), device=DEVICE, dtype=dtype)
y = torch.randn((m, m), device=DEVICE, dtype=dtype)
z = torch.empty((m, m), device=DEVICE, dtype=dtype)
gemm_samples = cuda_samples(lambda: torch.mm(x, y, out=z), repeats=20)
gemm_median = statistics.median(gemm_samples)
gemm_flops = 2 * m**3
gemm_tflops = gemm_flops / (gemm_median / 1e3) / 1e12
gemm_bytes = (x.numel() + y.numel() + z.numel()) * x.element_size()
metrics = {
    "vector_median_ms": vector_median,
    "vector_effective_gbps": vector_gbps,
    "vector_intensity_flop_byte": vector_intensity,
    "gemm_median_ms": gemm_median,
    "gemm_tflops": gemm_tflops,
    "gemm_intensity_flop_byte": gemm_flops / gemm_bytes,
    "vector_samples_ms": vector_samples,
    "gemm_samples_ms": gemm_samples,
}
analysis = (
    f"Vector addition delivered {vector_gbps:.1f} requested GB/s at only "
    f"{vector_intensity:.4f} FLOP/byte; the BF16 GEMM delivered {gemm_tflops:.1f} TFLOP/s "
    f"with an algorithmic intensity of {metrics['gemm_intensity_flop_byte']:.1f} FLOP/byte."
)
print(json.dumps(metrics, indent=2))


{
  "vector_median_ms": 0.25814399123191833,
  "vector_effective_gbps": 1559.8007223737918,
  "vector_intensity_flop_byte": 0.16666666666666666,
  "gemm_median_ms": 0.10432000085711479,
  "gemm_tflops": 164.68432748127515,
  "gemm_intensity_flop_byte": 682.6666666666666,
  "vector_samples_ms": [
    0.26044800877571106,
    0.26019200682640076,
    0.2588160037994385,
    0.25705599784851074,
    0.25731199979782104,
    0.26025599241256714,
    0.2581759989261627,
    0.25839999318122864,
    0.25702399015426636,
    0.25808000564575195,
    0.2568320035934448,
    0.2579520046710968,
    0.25628799200057983,
    0.25699201226234436,
    0.25814399123191833,
    0.25731199979782104,
    0.25833600759506226,
    0.2579199969768524,
    0.2593280076980591,
    0.2561280131340027,
    0.25939199328422546,
    0.25839999318122864,
    0.2561280131340027,
    0.25939199328422546,
    0.2592320144176483
  ],
  "gemm_samples_ms": [
    0.1111999973654747,
    0.10540799796581268,
    0.10499

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Vector median | 0.258 ms |
| Vector effective bandwidth | 1,559.8007 |
| GEMM median | 0.104 ms |
| GEMM throughput | 164.6843 |
| GEMM intensity | 682.6667 |


## 8. Explain rather than overclaim

Vector addition delivered 1559.8 requested GB/s at only 0.1667 FLOP/byte; the BF16 GEMM delivered 164.7 TFLOP/s with an algorithmic intensity of 682.7 FLOP/byte.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 4, "title": 'Why Data Movement Can Cost More Than Arithmetic', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Choose the next optimization from the limiting resource: reduce traffic for a bandwidth-bound kernel and improve math utilization only when compute is the credible ceiling.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 4,
  "title": "Why Data Movement Can Cost More Than Arithmetic",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260817
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "vector_median_ms": 0.25814399123191833,
    "vector_effective_gbps": 1559.8007223737918,
    "vector_intensity_flop_byte": 0.16666666666666666,
    "gemm_median_ms": 0.10432000085711479,
    "gemm_tflops": 164.68432748127515,
    "gemm_intensity_flop_byte": 682.6666666666666,
    "vector_samples_ms": [
      0.26044800877571106,
      0.26019200682640076,
      0.2588160037994385,
      0.25705599784851074,
      0.25731199979782104,
      0.26025599241256714,
      0.2581759989261627,
      0.25839999318122864,
      0.25702399015426636,
      0.25808000564575195,
      0.2568320035934448,
      0.2579520046710968,
      0.25628799200057983,
      0.256992012

## 10. Make the decision

> Choose the next optimization from the limiting resource: reduce traffic for a bandwidth-bound kernel and improve math utilization only when compute is the credible ceiling.

**Failure analysis:** Effective bandwidth is based on requested tensor bytes, not every physical transaction. GEMM may use implementation-specific precision and kernels. Shape changes can reverse the comparison.


## 11. Extend the evidence

Profile both operations with hierarchical Roofline counters and add a fused elementwise candidate that eliminates one intermediate write.

See [`README.md`](README.md) for the full explanation and references.
